In [24]:
from typing import Dict, List, Tuple

import pandas as pd
import torch
import transformers
from torch.utils.data import DataLoader, Dataset, random_split

In [25]:
df = pd.read_csv(r"..\data\databricks-dolly-15k.csv")

df

,instruction,context,response,category
0,When did Virgin Australia start operating?,"Virgin Australia, the trading name of Virgin A...",Virgin Australia commenced services on 31 Augu...,closed_qa
1,Which is a species of fish? Tope or Rope,NaN,Tope,classification
2,Why can camels survive for long without water?,NaN,Camels use the fat in their humps to keep them...,open_qa
3,"Alice's parents have three daughters: Amy, Jes...",NaN,The name of the third daughter is Alice,open_qa
4,When was Tomoaki Komorida born?,Komorida was born in Kumamoto Prefecture on Ju...,"Tomoaki Komorida was born on July 10,1981.",closed_qa
...,...,...,...,...
15010,How do i accept the change,NaN,Embrace the change and see the difference,brainstorming
15011,What is a laser and who created it?,A laser is a device that emits light through a...,A laser is a device that emits light from an e...,summarization
15012,What is the difference between a road bike and...,NaN,Road bikes are built to be ridden on asphalt a...,open_qa
15013,How does GIS help in the real estate investmen...,NaN,"Real estate investors depend on precise, accur...",general_qa


In [26]:
CATEGORY_MAP = {
    "general_qa": "q_and_a",
    "open_qa": "q_and_a",
    "closed_qa": "q_and_a",
    "information_extraction": "information_distillation",
    "summarization": "information_distillation",
}

In [27]:
df.isnull().sum()

instruction        0
context        10418
response           0
category           0
dtype: int64

In [28]:
def load_and_prepare_data(csv_path: str) -> Tuple[pd.DataFrame, Dict[str, int], Dict[int, str]]:
    """Loads the dataset, merges categories, and builds label mappings.

    Args:
        csv_path: path to the augmented Dolly-15k CSV. Must contain
            `instruction` and `category` columns.

    Returns:
        (df, cat2id, id2cat)
    """
    df = pd.read_csv(csv_path).dropna().reset_index(drop=True)

    if "instruction" not in df.columns or "category" not in df.columns:
        raise ValueError(
            f"Expected 'instruction' and 'category' columns, got {list(df.columns)}"
        )

    df["category"] = df["category"].replace(CATEGORY_MAP)

    unique_categories = df["category"].unique()
    cat2id = {category: i for i, category in enumerate(sorted(unique_categories))}
    id2cat = {i: category for category, i in cat2id.items()}

    df["label"] = df["category"].map(cat2id)

    return df, cat2id, id2cat

In [29]:
df, cat2id, id2cat = load_and_prepare_data(r"..\data\databricks-dolly-15k.csv")

df.isna().sum()

instruction    0
context        0
response       0
category       0
label          0
dtype: int64